# 🎓 3D Vision Studio: 2D Image to 3D Model AI Backend
### Fast 360° 3D Mesh Reconstruction Server powered by Google Colab GPU (T4)
### Model: **TripoSR** (100% Direct Download, Zero Tokens, Zero Hugging Face Logins)

---
### 🌟 Why TripoSR (Direct Download)?
- **Zero Logins & Zero Tokens**: 100% open weights directly downloaded via standard `wget`. No Hugging Face account or token needed!
- **Fast 360° Reconstruction**: Generates textured 3D meshes in ~2–3 seconds on a T4 GPU.
- **Watertight GLB Export**: Returns standard textured `.glb` models ready for Three.js, CAD, or 3D printing.

### 📌 Instructions:
1. Ensure you are connected to a **GPU Runtime**:
   - Click **Runtime** -> **Change runtime type** -> select **T4 GPU** -> **Save**.
2. Click **Runtime** -> **Run all** (or run each cell sequentially).
3. Cell 4 will output your public HTTPS URL (e.g. `https://xxxx.trycloudflare.com`).
4. Copy and paste that URL into your **3D Vision Studio** Web App settings!

In [ ]:
# Step 1: Verify NVIDIA GPU, Clone TripoSR & Direct-Download Model Weights
!nvidia-smi

print("[*] Cloning TripoSR repository...")
!git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR 2>/dev/null || (cd /content/TripoSR && git pull)

print("[*] Installing TripoSR requirements...")
!pip install -q -r /content/TripoSR/requirements.txt
!pip install -q git+https://github.com/tatsy/torchmcubes.git PyMCubes
!pip install -q rembg[gpu] trimesh fastapi uvicorn python-multipart

# Step 1.2: Direct Download model weights (No login, no token, 100% public direct link)
print("[*] Direct downloading TripoSR weights (~1.68 GB, zero tokens/login needed)...")
!mkdir -p /content/checkpoints
!wget -q -nc -O /content/checkpoints/config.yaml https://huggingface.co/stabilityai/TripoSR/resolve/main/config.yaml
!wget -q -nc -O /content/checkpoints/model.ckpt https://huggingface.co/stabilityai/TripoSR/resolve/main/model.ckpt

# Download cloudflared tunnel binary for instant public HTTPS endpoint
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("[+] All dependencies installed and model weights downloaded directly!")

In [ ]:
# Step 2: Initialize TripoSR Model from Local Checkpoint (0% HF connections)
import os
import sys
import torch

sys.path.append("/content/TripoSR")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"[*] Loading TripoSR onto: {device} from local disk...")

import rembg
from tsr.system import TSR

rembg_session = rembg.new_session()

# Load 100% offline from downloaded checkpoint
model = TSR.from_pretrained(
    "/content/checkpoints",
    config_name="config.yaml",
    weight_name="model.ckpt",
    is_local=True
)
model.renderer.set_chunk_size(8192)
model.to(device)

gpu_title = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"[+] TripoSR loaded successfully on: {gpu_title} (Completely offline & tokenless)!")

In [ ]:
# Step 3: Define FastAPI Endpoints (TripoSR Pipeline)
import io
import time
import numpy as np
import trimesh
from PIL import Image
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response

app = FastAPI(title="3D Vision Studio API (TripoSR)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def preprocess_image(input_image: Image.Image, foreground_ratio: float = 0.85) -> Image.Image:
    """Removes background, isolates foreground object, and normalizes into 512x512 with neutral gray."""
    raw_rgb = input_image.convert("RGB")
    img_rgba = rembg.remove(raw_rgb, session=rembg_session)
    bbox = img_rgba.getbbox()
    if bbox:
        img_rgba = img_rgba.crop(bbox)

    w, h = img_rgba.size
    max_side = max(w, h)
    target_box_size = int(max_side / foreground_ratio)
    padded = Image.new("RGBA", (target_box_size, target_box_size), (0, 0, 0, 0))
    paste_x = (target_box_size - w) // 2
    paste_y = (target_box_size - h) // 2
    padded.paste(img_rgba, (paste_x, paste_y))

    arr = np.array(padded.resize((512, 512), Image.Resampling.LANCZOS)).astype(np.float32) / 255.0
    rgb = arr[:, :, :3] * arr[:, :, 3:4] + (1.0 - arr[:, :, 3:4]) * 0.5
    return Image.fromarray((rgb * 255.0).astype(np.uint8))

@app.get("/health")
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_alloc = torch.cuda.memory_allocated(0) / (1024 ** 3) if torch.cuda.is_available() else 0.0
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0.0
    return {
        "status": "ok",
        "model": "TripoSR (Direct Download, Tokenless)",
        "gpu_name": gpu_name,
        "vram_allocated_gb": round(vram_alloc, 2),
        "vram_total_gb": round(vram_total, 2),
        "model_ready": True,
        "timestamp": time.time()
    }

@app.post("/api/generate")
async def generate_3d(image: UploadFile = File(...)):
    try:
        contents = await image.read()
        raw_img = Image.open(io.BytesIO(contents))

        # 1. Preprocess: rembg + center normalization
        proc_img = preprocess_image(raw_img)

        # 2. TripoSR Neural 3D synthesis (~2–3s)
        with torch.no_grad():
            scene_codes = model([proc_img], device=device)
            meshes = model.extract_mesh(scene_codes, has_vertex_color=True, resolution=256)
            mesh = meshes[0]

        # 3. Export binary GLB
        glb_io = io.BytesIO()
        mesh.export(glb_io, file_type="glb")

        return Response(
            content=glb_io.getvalue(),
            media_type="model/gltf-binary",
            headers={"Content-Disposition": 'attachment; filename="model.glb"'}
        )
    except Exception as e:
        print(f"[!] TripoSR generation error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

print("[+] TripoSR FastAPI Application defined.")

In [ ]:
# Step 4: Launch FastAPI Server & Expose via Cloudflare Tunnel
import subprocess
import threading
import time
import re
import uvicorn

# Start Uvicorn in background thread
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print("[+] Uvicorn server listening on port 8000.")

# Launch Cloudflared tunnel
print("[*] Launching secure Cloudflare public tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in iter(tunnel_proc.stdout.readline, ""):
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*60)
    print("🎉 SUCCESS! YOUR COLAB BACKEND IS ONLINE!")
    print(f"👉 COPY THIS URL INTO YOUR WEB APP:\n{tunnel_url}")
    print("="*60 + "\n")
else:
    print("[!] Cloudflare tunnel did not output URL yet. Check output above.")